# The Bias-Variance Tradeoff and the Double Descent Phenomenon
## An Interactive Lesson

**MAT 4953 / MAT 6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/capacity_U_graph.ipynb)

---

In this notebook we explore two of the most important conceptual frameworks for understanding generalization in machine learning:

1. **The classical bias-variance tradeoff** — a fundamental decomposition of prediction error that explains *why* models can fail, and reveals the characteristic **U-shaped test error curve** as model complexity increases.

2. **The modern under/overparametrization picture** — how highly overparametrized models (deep neural networks with far more parameters than training samples) can *still* generalize well, giving rise to a **double descent** curve that extends the classical U-shape.

We begin with toy polynomial regression to build geometric and algebraic intuition, then move to random feature models and neural networks where these phenomena become strikingly visible.

### Prerequisites

- Linear algebra (matrix rank, pseudoinverse, projection matrices)
- Probability (expectation, variance, mean squared error)
- Basic calculus and optimization
- Some familiarity with Python / NumPy

---
# 0. Setup

This notebook runs on **Google Colab** (GPU/CPU) and in a **local Jupyter environment**
(Linux, macOS, Windows).

- **Google Colab**: click *Open in Colab* above. Required packages are installed
  automatically by the setup cell.
- **Local**: install dependencies with `pip install -r requirements.txt` from the
  repository root, then open this file with `jupyter notebook` or `jupyter lab`.

The core computations use NumPy and scikit-learn (CPU only). The optional neural-network
section (Section 5) uses Keras; if Keras is not installed that section is skipped gracefully.

In [ ]:
# @title Environment setup (run first)
import sys, subprocess

# Reliable Colab detection
try:
    import google.colab  # noqa: F401
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'scikit-learn', 'matplotlib', 'ipywidgets'],
        check=False)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error
from IPython.display import display, HTML, Markdown

%matplotlib inline

# Reproducibility
np.random.seed(42)

print(f"Environment : {'Google Colab' if _in_colab else 'local'}")
print(f"NumPy version: {np.__version__}")


---
# 1. The Bias-Variance Decomposition

## 1.1 Setting and notation

Suppose we observe training data $(x_1, y_1), \dots, (x_n, y_n)$ generated by

$$y = f(x) + \varepsilon, \qquad \varepsilon \sim (0, \sigma^2),$$

where $f$ is an unknown **true function** and $\varepsilon$ is irreducible noise with mean zero and variance $\sigma^2$.

Given a learning algorithm $\mathcal{A}$, we train a model $\hat{f}$ on the data. Because the training set is random (through the noise $\varepsilon$ and possibly random sampling of $x$'s), the learned model $\hat{f}$ is itself a **random variable**.

## 1.2 The decomposition theorem

**Theorem (Bias-Variance Decomposition).** For any fixed test point $x_0$, the expected prediction error under squared loss decomposes as:

$$\mathbb{E}\bigl[(y_0 - \hat{f}(x_0))^2\bigr] \;=\; \underbrace{\bigl(f(x_0) - \mathbb{E}[\hat{f}(x_0)]\bigr)^2}_{\text{Bias}^2} \;+\; \underbrace{\mathbb{E}\bigl[(\hat{f}(x_0) - \mathbb{E}[\hat{f}(x_0)])^2\bigr]}_{\text{Variance}} \;+\; \underbrace{\sigma^2}_{\text{Irreducible noise}},$$

where all expectations are over the randomness in the training set (and the test noise $\varepsilon_0$).

*Proof sketch.* Write $y_0 = f(x_0) + \varepsilon_0$ and $\hat{f}(x_0) = \mathbb{E}[\hat{f}(x_0)] + (\hat{f}(x_0) - \mathbb{E}[\hat{f}(x_0)])$. Expanding the square and using independence of $\varepsilon_0$ from the training set gives the result. $\square$

### Intuition

| Term | Meaning | Increases when... |
|------|---------|-------------------|
| **Bias²** | How far the *average* model is from the truth | Model is too simple (underfitting) |
| **Variance** | How much the model *fluctuates* across training sets | Model is too complex (overfitting) |
| **Noise** | Irreducible — no model can do better than $\sigma^2$ | Always present |

## 1.3 Proof of the decomposition (detailed)

For the mathematically curious, here is the full calculation. Let $\mu(x_0) = \mathbb{E}[\hat{f}(x_0)]$.

$$
\begin{aligned}
\mathbb{E}[(y_0 - \hat{f}(x_0))^2]
&= \mathbb{E}[(f(x_0) + \varepsilon_0 - \hat{f}(x_0))^2] \\
&= \mathbb{E}[(f(x_0) - \hat{f}(x_0))^2] + \mathbb{E}[\varepsilon_0^2] + 2\,\mathbb{E}[(f(x_0) - \hat{f}(x_0))\varepsilon_0] \\
&= \mathbb{E}[(f(x_0) - \hat{f}(x_0))^2] + \sigma^2 \quad (\text{since } \varepsilon_0 \perp \hat{f})
\end{aligned}
$$

Now expand the first term by adding and subtracting $\mu(x_0)$:

$$
\begin{aligned}
\mathbb{E}[(f(x_0) - \hat{f}(x_0))^2]
&= \mathbb{E}[\bigl((f(x_0) - \mu(x_0)) + (\mu(x_0) - \hat{f}(x_0))\bigr)^2] \\
&= (f(x_0) - \mu(x_0))^2 + \mathbb{E}[(\hat{f}(x_0) - \mu(x_0))^2] + 2(f(x_0) - \mu(x_0))\underbrace{\mathbb{E}[\hat{f}(x_0) - \mu(x_0)]}_{=\,0} \\
&= \text{Bias}^2 + \text{Variance}.
\end{aligned}
$$

This completes the proof. $\square$

---
# 2. Toy Example: Polynomial Regression

## 2.1 The setup

We fit polynomials of increasing degree $d$ to noisy samples from $f(x) = \sin(x)$. A degree-$d$ polynomial has $d+1$ free parameters, so increasing $d$ increases model capacity.

This is the simplest setting where we can *see* the bias-variance tradeoff directly:
- **Low degree** (e.g., $d=1$): the line cannot capture the sine wave → high bias, low variance.
- **High degree** (e.g., $d=15$): the polynomial wiggles wildly through noisy points → low bias, high variance.
- **Just right** (e.g., $d=3$ or $4$): a good compromise.

In [ ]:
# --- Polynomial Regression: Visual Exploration ---

N_SAMPLES = 20
NOISE_LEVEL = 0.4

def generate_data(n=N_SAMPLES, noise=NOISE_LEVEL, seed=None):
    """Generate noisy samples from f(x) = sin(x)."""
    if seed is not None:
        rng = np.random.RandomState(seed)
    else:
        rng = np.random.RandomState()
    X = np.sort(rng.uniform(0, 2 * np.pi, n))
    y_true = np.sin(X)
    y_noisy = y_true + rng.normal(0, noise, n)
    return X, y_true, y_noisy

x_plot = np.linspace(0, 2 * np.pi, 300)

# Show three fits side by side: underfitting, good fit, overfitting
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
degrees = [1, 4, 15]
titles = ['Underfitting (d=1)', 'Good fit (d=4)', 'Overfitting (d=15)']

X_demo, y_true_demo, y_noisy_demo = generate_data(seed=42)

for ax, deg, title in zip(axes, degrees, titles):
    model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    model.fit(X_demo.reshape(-1, 1), y_noisy_demo)
    y_pred = model.predict(x_plot.reshape(-1, 1))
    
    ax.scatter(X_demo, y_noisy_demo, c='steelblue', edgecolors='k', s=40, zorder=5, label='Training data')
    ax.plot(x_plot, np.sin(x_plot), 'k--', alpha=0.5, label=r'True $f(x)=\sin(x)$')
    ax.plot(x_plot, y_pred, 'r-', linewidth=2, label=f'Degree {deg} fit')
    ax.set_ylim(-2.5, 2.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Polynomial Regression: Three Regimes', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Observations:**
- The degree-1 polynomial (a line) systematically misses the curvature of $\sin(x)$ — this is **bias**.
- The degree-15 polynomial passes near every noisy point but oscillates wildly between them — this is **variance**.
- The degree-4 polynomial captures the overall shape without chasing noise.

## 2.2 Measuring bias and variance empirically

The bias-variance decomposition involves expectations over *many possible training sets*. We can estimate these by Monte Carlo simulation:

1. Generate $M$ independent training sets (same size, different noise draws).
2. Fit the model on each training set.
3. At each test point $x_0$, compute:
   - $\widehat{\text{Bias}}^2(x_0) = \bigl(f(x_0) - \bar{\hat{f}}(x_0)\bigr)^2$ where $\bar{\hat{f}}$ is the average prediction.
   - $\widehat{\text{Var}}(x_0) = \frac{1}{M}\sum_{m=1}^{M}(\hat{f}_m(x_0) - \bar{\hat{f}}(x_0))^2$.

In [ ]:
# --- Monte Carlo Bias-Variance Estimation ---

M_SIMULATIONS = 200   # Number of training sets to draw
MAX_DEGREE = 18
x_test = np.linspace(0.2, 2 * np.pi - 0.2, 50)  # Avoid boundary effects
f_true_test = np.sin(x_test)

bias2_by_degree = []
var_by_degree = []
mse_by_degree = []

for deg in range(1, MAX_DEGREE + 1):
    predictions = np.zeros((M_SIMULATIONS, len(x_test)))
    
    for m in range(M_SIMULATIONS):
        X_m, _, y_m = generate_data(seed=1000 + m)
        model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
        model.fit(X_m.reshape(-1, 1), y_m)
        predictions[m, :] = model.predict(x_test.reshape(-1, 1))
    
    # Clip extreme predictions for numerical stability
    predictions = np.clip(predictions, -10, 10)
    
    mean_pred = predictions.mean(axis=0)
    bias2 = np.mean((f_true_test - mean_pred) ** 2)
    var = np.mean(predictions.var(axis=0))
    mse = bias2 + var + NOISE_LEVEL**2
    
    bias2_by_degree.append(bias2)
    var_by_degree.append(var)
    mse_by_degree.append(mse)

degrees_range = np.arange(1, MAX_DEGREE + 1)

print("Monte Carlo simulation complete.")
print(f"  Optimal degree (min MSE): {degrees_range[np.argmin(mse_by_degree)]}")

In [ ]:
# --- Plot the U-shaped curve ---

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(degrees_range, bias2_by_degree, 'b--o', markersize=5, label=r'Bias$^2$', linewidth=2)
ax.plot(degrees_range, var_by_degree, 'g--s', markersize=5, label='Variance', linewidth=2)
ax.fill_between(degrees_range, 0, NOISE_LEVEL**2, alpha=0.15, color='gray', label=r'Irreducible noise $\sigma^2$')
ax.plot(degrees_range, mse_by_degree, 'r-D', markersize=6, label='Total test error (MSE)', linewidth=2.5)

# Mark optimal
opt_idx = np.argmin(mse_by_degree)
ax.axvline(degrees_range[opt_idx], color='gray', linestyle=':', alpha=0.7)
ax.annotate(f'Optimal (d={degrees_range[opt_idx]})',
            xy=(degrees_range[opt_idx], mse_by_degree[opt_idx]),
            xytext=(degrees_range[opt_idx] + 2, mse_by_degree[opt_idx] + 0.3),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=11, color='gray')

# Shaded regions
ax.axvspan(0.5, degrees_range[opt_idx], alpha=0.06, color='blue')
ax.axvspan(degrees_range[opt_idx], MAX_DEGREE + 0.5, alpha=0.06, color='red')
ax.text(2, max(mse_by_degree) * 0.85, 'UNDERFITTING\n(high bias)', fontsize=11, color='blue', ha='center', alpha=0.7)
ax.text(MAX_DEGREE - 2, max(mse_by_degree) * 0.85, 'OVERFITTING\n(high variance)', fontsize=11, color='red', ha='center', alpha=0.7)

ax.set_xlabel('Polynomial degree (model complexity)', fontsize=12)
ax.set_ylabel('Error', fontsize=12)
ax.set_title('The Classical U-Shaped Test Error Curve', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0.5, MAX_DEGREE + 0.5)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**The U-Shape Explained:**

As polynomial degree $d$ increases:
- **Bias² decreases** because the model class becomes richer and can approximate $f$ more closely.
- **Variance increases** because more parameters means more sensitivity to the particular training sample.
- **Total test error** first decreases (bias reduction dominates) then increases (variance dominates), forming the characteristic **U-shape**.

The minimum of this U-curve represents the **optimal model complexity** — the best tradeoff between underfitting and overfitting.

## 2.3 Visualizing the ensemble of fits

To build more intuition, let us overlay many fitted curves (from different training sets) for three different polynomial degrees. This makes the *variance* directly visible as the spread of the fitted curves.

In [ ]:
# --- Ensemble of polynomial fits ---

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
degrees_show = [1, 4, 15]
n_show = 30  # Number of fits to overlay

for ax, deg in zip(axes, degrees_show):
    all_preds = []
    for m in range(n_show):
        X_m, _, y_m = generate_data(seed=2000 + m)
        model = make_pipeline(PolynomialFeatures(deg), LinearRegression())
        model.fit(X_m.reshape(-1, 1), y_m)
        y_pred = np.clip(model.predict(x_plot.reshape(-1, 1)), -3, 3)
        ax.plot(x_plot, y_pred, color='red', alpha=0.15, linewidth=0.8)
        all_preds.append(y_pred)
    
    # Average prediction
    mean_pred = np.mean(all_preds, axis=0)
    ax.plot(x_plot, mean_pred, 'r-', linewidth=2.5, label=r'$\bar{\hat{f}}$ (mean prediction)')
    ax.plot(x_plot, np.sin(x_plot), 'k--', linewidth=2, label=r'$f(x) = \sin(x)$')
    
    ax.set_ylim(-3, 3)
    ax.set_title(f'Degree {deg}: {"High bias" if deg == 1 else "Balanced" if deg == 4 else "High variance"}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)

fig.suptitle('Ensemble of Fits from Different Training Sets', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**What to observe:**
- **Degree 1 (left):** All lines are similar (low variance) but they *all* miss the true function (high bias). The red mean $\bar{\hat{f}}$ is far from the black dashed truth.
- **Degree 4 (center):** Moderate spread of curves, and the mean tracks the truth closely. Good tradeoff.
- **Degree 15 (right):** The individual curves are wildly different from each other (high variance), even though their average may be close to the truth.

---
# 3. The Interpolation Threshold and the Linear Algebra Perspective

## 3.1 When does a model interpolate exactly?

Consider a linear model $\hat{f}(x) = \boldsymbol{\phi}(x)^\top \boldsymbol{\beta}$ with $p$ features (parameters). When we fit it to $n$ training points using least squares, we solve

$$\boldsymbol{\Phi} \boldsymbol{\beta} = \mathbf{y},$$

where $\boldsymbol{\Phi} \in \mathbb{R}^{n \times p}$ is the feature matrix and $\boldsymbol{\beta} \in \mathbb{R}^p$ is the coefficient vector.

Three regimes emerge based on the relationship between $p$ and $n$:

| Regime | Condition | System | Training error |
|--------|-----------|--------|----------------|
| **Underparametrized** | $p < n$ | Overdetermined | $> 0$ (cannot interpolate) |
| **Interpolation threshold** | $p = n$ | Square (unique solution) | $= 0$ (exact interpolation) |
| **Overparametrized** | $p > n$ | Underdetermined | $= 0$ (infinitely many solutions) |

For polynomial regression with the Vandermonde matrix $V_{ij} = x_i^{j-1}$, the interpolation threshold occurs at degree $d = n - 1$ (giving $p = n$ parameters).

## 3.2 The minimum-norm solution

When $p > n$ (overparametrized), there are infinitely many $\boldsymbol{\beta}$ that achieve zero training error. Ordinary least squares (via the pseudoinverse) selects the **minimum-norm** solution:

$$\hat{\boldsymbol{\beta}} = \boldsymbol{\Phi}^\dagger \mathbf{y} = \boldsymbol{\Phi}^\top (\boldsymbol{\Phi} \boldsymbol{\Phi}^\top)^{-1} \mathbf{y}.$$

This minimum-norm inductive bias is crucial: it acts as an *implicit regularizer* that can prevent overfitting even when the model has far more parameters than data points.

## 3.3 Why does the interpolation threshold matter?

At $p = n$, the system $\boldsymbol{\Phi}\boldsymbol{\beta} = \mathbf{y}$ has a *unique* solution. The solution norm is

$$\|\hat{\boldsymbol{\beta}}\|^2 = \mathbf{y}^\top (\boldsymbol{\Phi}\boldsymbol{\Phi}^\top)^{-1} \mathbf{y},$$

and the smallest eigenvalue of $\boldsymbol{\Phi}\boldsymbol{\Phi}^\top$ approaches zero as $p \to n$. This makes the matrix nearly singular, **amplifying noise** and producing large, oscillatory coefficients.

In [ ]:
# --- Training error vs. test error across the interpolation threshold ---
# Using polynomial regression with a small dataset

np.random.seed(42)
n = 15  # Small n so we can cross the threshold easily
X_small = np.sort(np.random.uniform(0.1, 2 * np.pi - 0.1, n))
y_small = np.sin(X_small) + np.random.normal(0, 0.4, n)

x_test_dense = np.linspace(0.1, 2 * np.pi - 0.1, 200)
y_test_true = np.sin(x_test_dense)

max_d = 25
train_errors = []
test_errors = []
n_params_list = []

for d in range(1, max_d + 1):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(X_small.reshape(-1, 1), y_small)
    
    y_train_pred = model.predict(X_small.reshape(-1, 1))
    y_test_pred = np.clip(model.predict(x_test_dense.reshape(-1, 1)), -5, 5)
    
    train_errors.append(mean_squared_error(y_small, y_train_pred))
    test_errors.append(mean_squared_error(y_test_true, y_test_pred))
    n_params_list.append(d + 1)

fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(n_params_list, train_errors, 'o-', color='#ff7f0e', label='Training error', markersize=5)
ax.semilogy(n_params_list, test_errors, 's-', color='#d62728', label='Test error', markersize=5)
ax.axvline(n, color='purple', linestyle='--', linewidth=2, alpha=0.7, label=f'Interpolation threshold ($p = n = {n}$)')

ax.set_xlabel('Number of parameters ($p = d + 1$)', fontsize=12)
ax.set_ylabel('Mean Squared Error (log scale)', fontsize=12)
ax.set_title('Training and Test Error Across the Interpolation Threshold', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(1, max_d + 1)

plt.tight_layout()
plt.show()

**Observations:**
- **Training error** drops to (near) zero once $p \geq n$ — the model can interpolate the data.
- **Test error** often *peaks* near $p = n$ (the interpolation threshold), where the model barely has enough capacity to memorize the data, including its noise.

---
# 4. The Double Descent Curve

## 4.1 Beyond the classical picture

The classical bias-variance tradeoff predicts that test error should only *increase* as we add parameters beyond the optimum. But modern deep learning practice contradicts this: networks with **millions or billions** of parameters (far more than training examples) often generalize *better* than smaller networks.

The **double descent** curve, studied by Belkin et al. (2019) and others, reconciles these observations:

$$\text{Test error as a function of } p/n: \qquad
\underbrace{\searrow}_{\text{classical}} \;\; 
\underbrace{\nearrow}_{\text{peak at } p \approx n} \;\;
\underbrace{\searrow}_{\text{modern interpolating regime}}
$$

The curve has two "descents":
1. **First descent** (classical regime, $p < n$): More parameters help reduce bias.
2. **Peak** (interpolation threshold, $p \approx n$): The model barely interpolates — it fits *every* noise sample exactly, with no "room" for a smooth solution.
3. **Second descent** (overparametrized regime, $p \gg n$): Among the many interpolating solutions, the minimum-norm one is increasingly smooth.

## 4.2 A clean demonstration with random features

To demonstrate double descent unambiguously, we use a **random feature model** with Gaussian design. This is the cleanest theoretical setting and the one analyzed rigorously in the literature.

**Setup:** The true model lives in $\mathbb{R}^{d_0}$ (a low-dimensional signal space). We augment the input with additional random (noise) features to control the total number of parameters $p$:

- For $p \leq d_0$: use the first $p$ signal features.
- For $p > d_0$: use all $d_0$ signal features plus $(p - d_0)$ random noise features.

We then fit the minimum-norm least-squares solution and measure test error.

In [ ]:
# --- Double Descent with Random Gaussian Features ---

np.random.seed(42)

d_true = 10   # True signal dimension
n_dd = 50     # Number of training samples
sigma_dd = 0.5  # Noise level

# True model: beta_true in R^{d_true}
beta_true = np.random.randn(d_true) / np.sqrt(d_true)

# Generate training data
X_base_train = np.random.randn(n_dd, d_true)
y_train_dd = X_base_train @ beta_true + sigma_dd * np.random.randn(n_dd)

# Generate test data
n_test_dd = 500
X_base_test = np.random.randn(n_test_dd, d_true)
y_test_dd_true = X_base_test @ beta_true + sigma_dd * np.random.randn(n_test_dd)

# Sweep over number of parameters p
p_values = list(range(2, 300, 2))

# Average over multiple random feature draws for stability
n_feature_seeds = 10
avg_train_err = np.zeros(len(p_values))
avg_test_err = np.zeros(len(p_values))

for seed in range(n_feature_seeds):
    rng = np.random.RandomState(seed * 100 + 7)
    for ip, p in enumerate(p_values):
        if p <= d_true:
            # Use first p signal features
            X_tr = X_base_train[:, :p]
            X_te = X_base_test[:, :p]
        else:
            # Signal features + random noise features
            extra_train = rng.randn(n_dd, p - d_true) / np.sqrt(p)
            extra_test = rng.randn(n_test_dd, p - d_true) / np.sqrt(p)
            X_tr = np.hstack([X_base_train, extra_train])
            X_te = np.hstack([X_base_test, extra_test])
        
        # Minimum-norm least squares via pseudoinverse
        beta_hat, _, _, _ = np.linalg.lstsq(X_tr, y_train_dd, rcond=None)
        
        y_tr_pred = X_tr @ beta_hat
        y_te_pred = X_te @ beta_hat
        
        avg_train_err[ip] += mean_squared_error(y_train_dd, y_tr_pred)
        avg_test_err[ip] += mean_squared_error(y_test_dd_true, y_te_pred)

avg_train_err /= n_feature_seeds
avg_test_err /= n_feature_seeds

print(f"Signal dimension: d_0 = {d_true}")
print(f"Training samples: n = {n_dd}")
print(f"Interpolation threshold: p = n = {n_dd}")
print(f"Peak test error at p = {p_values[np.argmax(avg_test_err)]}")
print(f"Min test error (overparametrized): p = {p_values[np.argmin(avg_test_err[n_dd//2:]) + n_dd//2]}")

In [ ]:
# --- Plot the double descent curve ---

fig, ax = plt.subplots(figsize=(11, 6))

ax.semilogy(p_values, avg_train_err, 'o-', color='#ff7f0e', markersize=2, label='Training error', alpha=0.8)
ax.semilogy(p_values, avg_test_err, 's-', color='#d62728', markersize=2, label='Test error', linewidth=2)

ax.axvline(n_dd, color='purple', linestyle='--', linewidth=2, alpha=0.7,
           label=f'Interpolation threshold ($p = n = {n_dd}$)')

# Annotate the three regimes
ymax = max(avg_test_err) * 2
ax.axvspan(0, n_dd, alpha=0.05, color='blue')
ax.axvspan(n_dd, max(p_values), alpha=0.05, color='green')

ax.text(n_dd / 2, ymax * 0.3, 'UNDERPARAMETRIZED\n$p < n$', ha='center', fontsize=11, color='blue', alpha=0.6)
ax.text((n_dd + max(p_values)) / 2, ymax * 0.3, 'OVERPARAMETRIZED\n$p > n$', ha='center', fontsize=11, color='green', alpha=0.6)

# Annotate the peak
peak_idx = np.argmax(avg_test_err)
ax.annotate(f'Peak at $p \\approx n$\n(test error = {avg_test_err[peak_idx]:.1f})',
            xy=(p_values[peak_idx], avg_test_err[peak_idx]),
            xytext=(p_values[peak_idx] + 40, avg_test_err[peak_idx] * 0.5),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')

ax.set_xlabel('Number of parameters $p$', fontsize=12)
ax.set_ylabel('Mean Squared Error (log scale)', fontsize=12)
ax.set_title('Double Descent: Test Error with Random Gaussian Features', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(1, max(p_values))

plt.tight_layout()
plt.show()

**The Double Descent Explained:**

1. **First descent** ($p < n$): Adding parameters improves the fit by reducing bias.
2. **Peak** ($p \approx n$): The model can just barely interpolate. The unique solution amplifies noise because $(\boldsymbol{\Phi}^\top\boldsymbol{\Phi})^{-1}$ has enormous eigenvalues. Test error can spike by orders of magnitude.
3. **Second descent** ($p \gg n$): Among the infinitely many interpolating solutions, the minimum-norm one has $\|\hat{\boldsymbol{\beta}}\|^2 = \mathbf{y}^\top(\boldsymbol{\Phi}\boldsymbol{\Phi}^\top)^{-1}\mathbf{y}$, which *decreases* as the eigenvalues of $\boldsymbol{\Phi}\boldsymbol{\Phi}^\top$ grow with $p$. The implicit $\ell^2$ regularization becomes stronger.

This is sometimes called **benign overfitting**: the model memorizes the training data (zero training error) yet still predicts well on unseen data.

## 4.3 The condition number tells the story

Let us verify our mathematical explanation by plotting the condition number of $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ (or $\boldsymbol{\Phi}\boldsymbol{\Phi}^\top$ in the overparametrized regime). We expect it to blow up near $p = n$.

In [ ]:
# --- Condition number analysis ---

rng_cond = np.random.RandomState(42)
cond_numbers = []
solution_norms = []

for p in p_values:
    if p <= d_true:
        X_tr = X_base_train[:, :p]
    else:
        extra = rng_cond.randn(n_dd, p - d_true) / np.sqrt(p)
        X_tr = np.hstack([X_base_train, extra])
    
    # Condition number
    s = np.linalg.svd(X_tr, compute_uv=False)
    cond = s[0] / max(s[-1], 1e-15)
    cond_numbers.append(cond)
    
    # Solution norm
    beta_hat, _, _, _ = np.linalg.lstsq(X_tr, y_train_dd, rcond=None)
    solution_norms.append(np.linalg.norm(beta_hat))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogy(p_values, cond_numbers, 'b-', linewidth=1.5)
ax1.axvline(n_dd, color='purple', linestyle='--', linewidth=2, alpha=0.7, label=f'$p = n = {n_dd}$')
ax1.set_xlabel('Number of parameters $p$', fontsize=12)
ax1.set_ylabel(r'Condition number $\kappa(\mathbf{\Phi})$', fontsize=12)
ax1.set_title('Condition Number Diverges at $p = n$', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, which='both')

ax2.semilogy(p_values, solution_norms, 'r-', linewidth=1.5)
ax2.axvline(n_dd, color='purple', linestyle='--', linewidth=2, alpha=0.7, label=f'$p = n = {n_dd}$')
ax2.set_xlabel('Number of parameters $p$', fontsize=12)
ax2.set_ylabel(r'Solution norm $\|\hat{\beta}\|$', fontsize=12)
ax2.set_title('Solution Norm Peaks at $p = n$', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

**Key observations:**
- The condition number $\kappa(\boldsymbol{\Phi})$ spikes dramatically at $p = n$, confirming that the linear system becomes nearly singular.
- The solution norm $\|\hat{\boldsymbol{\beta}}\|$ also peaks at $p = n$ and then *decreases* in the overparametrized regime. This is the **implicit regularization** of the minimum-norm interpolator: more parameters lead to smaller coefficient norms and smoother solutions.

---
# 5. The Interactive U-Curve Explorer

Use the slider below to explore the bias-variance tradeoff interactively. The vertical black line shows your selected complexity level, and the info box displays the current decomposition values.

In [ ]:
# --- Interactive Bias-Variance U-Curve ---
from ipywidgets import FloatSlider, VBox, HTML, interactive_output, Layout

# Generate smooth analytic-style curves for the interactive plot
x_ax = np.linspace(0, 100, 500)
bias_sq_curve = 8000 / (x_ax + 20)**1.5
var_curve = 0.05 * x_ax**1.8
noise_curve = np.full_like(x_ax, 15)
test_err_curve = bias_sq_curve + var_curve + noise_curve
train_err_curve = 7000 / (x_ax + 20)**1.6
opt_x = x_ax[np.argmin(test_err_curve)]

def plot_bv_interactive(complexity):
    fig, ax = plt.subplots(figsize=(10, 6))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.08)
    
    ax.plot(x_ax, bias_sq_curve, label=r'Bias$^2$', color='#1f77b4', linestyle='--', linewidth=2)
    ax.plot(x_ax, var_curve, label='Variance', color='#2ca02c', linestyle='--', linewidth=2)
    ax.plot(x_ax, train_err_curve, label='Training error', color='#ff7f0e', linewidth=1.5)
    ax.plot(x_ax, test_err_curve, label='Total test error', color='#d62728', linewidth=2.5)
    
    # Optimal line
    ax.axvline(opt_x, color='gray', linestyle=':', alpha=0.7, label='Optimal complexity')
    
    # Shaded regions
    ax.axvspan(0, opt_x, alpha=0.08, color='blue', label='Underfitting zone')
    ax.axvspan(opt_x, 100, alpha=0.08, color='red', label='Overfitting zone')
    
    # Current complexity line
    ax.axvline(complexity, color='black', linestyle='-', linewidth=1.5, alpha=0.8)
    
    # Extract current values
    idx = np.argmin(np.abs(x_ax - complexity))
    b2 = bias_sq_curve[idx]
    v = var_curve[idx]
    tr = train_err_curve[idx]
    te = test_err_curve[idx]
    
    textstr = (f'Complexity: {complexity:.0f}\n'
               f'───────────────\n'
               f'Bias²:       {b2:.1f}\n'
               f'Variance:    {v:.1f}\n'
               f'Train err:   {tr:.1f}\n'
               f'Test err:    {te:.1f}')
    ax.text(0.02, 0.97, textstr, transform=ax.transAxes, verticalalignment='top',
            fontfamily='monospace', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))
    
    ax.set_title('The Bias-Variance Tradeoff (Interactive)', fontsize=14, fontweight='bold')
    ax.set_xlabel('Model Complexity', fontsize=12)
    ax.set_ylabel('Error', fontsize=12)
    ax.set_ylim(0, np.max(test_err_curve) + 10)
    ax.set_xlim(0, 100)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.show()

complexity_slider = FloatSlider(
    value=50, min=0, max=100, step=0.5,
    description='', readout=False,
    style={'description_width': '0px'},
    layout=Layout(width='100%', margin='0px'),
    continuous_update=True
)

out = interactive_output(plot_bv_interactive, {'complexity': complexity_slider})
slider_label = HTML('<div style="text-align:center; font-weight:bold;">Drag to explore model complexity</div>')
display(VBox([out, complexity_slider, slider_label]))

**Experiment:** Drag the slider to observe:
- At low complexity: Bias² dominates, variance is small.
- At high complexity: Variance dominates, bias² is small.
- At the optimal point: Test error is minimized.

---
# 6. Neural Networks: Double Descent in Practice

## 6.1 Model-wise double descent

The double descent phenomenon is not limited to toy linear models. It has been observed in deep neural networks trained on standard benchmarks. We demonstrate this using small fully-connected networks on a subset of MNIST, varying the **network width** (number of hidden units) to control the parameter count.

To make the interpolation threshold peak more dramatic, we intentionally add **label noise**: a fraction of training labels are randomized. This forces the network to memorize arbitrary label assignments, amplifying the effect near the threshold.

In [ ]:
# --- Neural Network Double Descent on MNIST subset ---

try:
    import keras
    from keras import layers
    HAS_KERAS = True
    print(f"Keras available (version {keras.__version__})")
except ImportError:
    try:
        import tensorflow as tf
        from tensorflow import keras
        from tensorflow.keras import layers
        HAS_KERAS = True
        print(f"TensorFlow/Keras available (version {tf.__version__})")
    except ImportError:
        HAS_KERAS = False
        print("Keras/TensorFlow not available. Skipping neural network experiments.")
        print("To install on Colab: these are pre-installed.")
        print("To install locally: pip install keras tensorflow")

In [ ]:
if HAS_KERAS:
    # Load MNIST and take a small subset
    (x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.mnist.load_data()
    
    # Use only digits 0-4 and a small training set to reach interpolation
    n_train = 500
    n_test_nn = 1000
    n_classes = 5
    
    # Filter to first n_classes digits
    train_mask = y_train_full < n_classes
    test_mask = y_test_full < n_classes
    
    x_tr_nn = x_train_full[train_mask][:n_train].reshape(-1, 784).astype('float32') / 255.0
    y_tr_nn = y_train_full[train_mask][:n_train]
    x_te_nn = x_test_full[test_mask][:n_test_nn].reshape(-1, 784).astype('float32') / 255.0
    y_te_nn = y_test_full[test_mask][:n_test_nn]
    
    # Add label noise to make the interpolation threshold more dramatic
    noise_frac = 0.15
    n_noisy = int(noise_frac * n_train)
    rng_nn = np.random.RandomState(42)
    noisy_idx = rng_nn.choice(n_train, n_noisy, replace=False)
    y_tr_noisy = y_tr_nn.copy()
    y_tr_noisy[noisy_idx] = rng_nn.randint(0, n_classes, n_noisy)
    
    print(f"Training set: {n_train} samples, {n_classes} classes")
    print(f"Test set: {n_test_nn} samples")
    print(f"Label noise: {noise_frac*100:.0f}% of training labels randomized")
    print(f"Input dimension: {x_tr_nn.shape[1]}")
else:
    print("Skipping (Keras not available).")

In [ ]:
if HAS_KERAS:
    import warnings
    warnings.filterwarnings('ignore')
    
    def count_params(model):
        return sum(p.size for p in model.get_weights())
    
    def train_and_evaluate(width, x_train, y_train, x_test, y_test, n_classes, epochs=200):
        """Train a 1-hidden-layer network with given width."""
        model = keras.Sequential([
            layers.Input(shape=(784,)),
            layers.Dense(width, activation='relu'),
            layers.Dense(n_classes, activation='softmax')
        ])
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        
        history = model.fit(x_train, y_train, epochs=epochs, batch_size=min(64, len(x_train)),
                           verbose=0, validation_data=(x_test, y_test))
        
        train_loss, train_acc = model.evaluate(x_train, y_train, verbose=0)
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
        n_p = count_params(model)
        
        return n_p, train_loss, test_loss, train_acc, test_acc
    
    # Sweep over network widths
    widths = [2, 4, 6, 8, 10, 15, 20, 30, 50, 75, 100] # 100 is already very large for this tiny dataset.
    results = []
    
    print("Training networks of varying width (this may take a few minutes)...")
    for w in widths:
        n_p, tr_loss, te_loss, tr_acc, te_acc = train_and_evaluate(
            w, x_tr_nn, y_tr_noisy, x_te_nn, y_te_nn, n_classes, epochs=300
        )
        results.append((w, n_p, tr_loss, te_loss, tr_acc, te_acc))
        print(f"  width={w:>5d}  params={n_p:>8d}  train_acc={tr_acc:.3f}  test_acc={te_acc:.3f}")
    
    nn_widths, nn_params, nn_tr_loss, nn_te_loss, nn_tr_acc, nn_te_acc = zip(*results)
    print("\nDone!")
else:
    print("Skipping (Keras not available).")

In [ ]:
if HAS_KERAS:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Plot 1: Loss
    ax1.semilogy(nn_params, nn_tr_loss, 'o-', color='#ff7f0e', label='Training loss', markersize=6)
    ax1.semilogy(nn_params, nn_te_loss, 's-', color='#d62728', label='Test loss', markersize=6, linewidth=2)
    ax1.axvline(n_train * n_classes, color='purple', linestyle='--', alpha=0.5,
               label=f'~Interpolation threshold\n({n_train} x {n_classes} = {n_train*n_classes})')
    ax1.set_xlabel('Number of parameters', fontsize=12)
    ax1.set_ylabel('Loss (log scale)', fontsize=12)
    ax1.set_title('Model-wise Double Descent (Loss)', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3, which='both')
    ax1.set_xscale('log')
    
    # Plot 2: Error rate
    nn_te_err = [1 - a for a in nn_te_acc]
    nn_tr_err = [1 - a for a in nn_tr_acc]
    ax2.plot(nn_params, nn_tr_err, 'o-', color='#ff7f0e', label='Training error', markersize=6)
    ax2.plot(nn_params, nn_te_err, 's-', color='#d62728', label='Test error', markersize=6, linewidth=2)
    ax2.axvline(n_train * n_classes, color='purple', linestyle='--', alpha=0.5,
               label='~Interpolation threshold')
    ax2.set_xlabel('Number of parameters', fontsize=12)
    ax2.set_ylabel('Classification error rate', fontsize=12)
    ax2.set_title('Model-wise Double Descent (Error)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    ax2.set_xscale('log')
    
    plt.tight_layout()
    plt.show()
else:
    print("Skipping (Keras not available).")

**Observations:**
- As network width increases, the number of parameters grows and the network's capacity to memorize increases.
- Near the interpolation threshold (where the number of effective parameters is roughly $n \times C$), test error can spike.
- Beyond this threshold, adding more parameters *improves* test performance — the second descent.
- Label noise amplifies the effect: the model must memorize both signal and noise, making the threshold peak sharper.

---
# 7. Regularization and Its Effect on the U-Curve

## 7.1 Ridge regression as explicit regularization

Regularization is the classical remedy for overfitting. **Ridge regression** ($\ell^2$ penalty) replaces the least-squares objective with

$$\hat{\boldsymbol{\beta}}_{\text{ridge}} = \arg\min_{\boldsymbol{\beta}} \|\boldsymbol{\Phi}\boldsymbol{\beta} - \mathbf{y}\|^2 + \lambda \|\boldsymbol{\beta}\|^2,$$

which has the closed-form solution $\hat{\boldsymbol{\beta}}_{\text{ridge}} = (\boldsymbol{\Phi}^\top \boldsymbol{\Phi} + \lambda \mathbf{I})^{-1} \boldsymbol{\Phi}^\top \mathbf{y}$.

The penalty $\lambda > 0$ shrinks the coefficients, reducing variance at the cost of increased bias. Crucially:

$$\kappa(\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda \mathbf{I}) \leq \frac{\sigma_1^2 + \lambda}{\sigma_n^2 + \lambda} \ll \kappa(\boldsymbol{\Phi}^\top\boldsymbol{\Phi}) \quad \text{when } \sigma_n^2 \approx 0.$$

The regularization *floors* the smallest eigenvalues, preventing the condition number from diverging and **eliminating the interpolation threshold peak**.

In [ ]:
# --- Regularization smooths the double descent peak ---

def fit_ridge_random_features(X_base_train, y_train, X_base_test, y_test_true,
                               p, alpha, d_true, rng):
    """Fit random feature model with Ridge regularization."""
    n = len(y_train)
    n_te = len(y_test_true)
    
    if p <= d_true:
        X_tr = X_base_train[:, :p]
        X_te = X_base_test[:, :p]
    else:
        extra_train = rng.randn(n, p - d_true) / np.sqrt(p)
        extra_test = rng.randn(n_te, p - d_true) / np.sqrt(p)
        X_tr = np.hstack([X_base_train, extra_train])
        X_te = np.hstack([X_base_test, extra_test])
    
    if alpha > 0:
        ridge = Ridge(alpha=alpha, fit_intercept=False)
        ridge.fit(X_tr, y_train)
        return mean_squared_error(y_test_true, ridge.predict(X_te))
    else:
        beta_hat, _, _, _ = np.linalg.lstsq(X_tr, y_train, rcond=None)
        return mean_squared_error(y_test_true, X_te @ beta_hat)

# Compare unregularized vs. regularized
alphas = [0, 0.01, 0.1, 1.0]
colors_reg = ['#d62728', '#9467bd', '#2ca02c', '#1f77b4']
n_seeds_reg = 8

fig, ax = plt.subplots(figsize=(11, 6))

for alpha, color in zip(alphas, colors_reg):
    test_errs_avg = np.zeros(len(p_values))
    for seed in range(n_seeds_reg):
        rng_r = np.random.RandomState(seed * 100 + 7)
        for ip, p in enumerate(p_values):
            test_errs_avg[ip] += fit_ridge_random_features(
                X_base_train, y_train_dd, X_base_test, y_test_dd_true,
                p, alpha, d_true, rng_r)
    test_errs_avg /= n_seeds_reg
    
    label = f'$\\lambda = {alpha}$' + (' (no regularization)' if alpha == 0 else '')
    ax.semilogy(p_values, test_errs_avg, '-', color=color, linewidth=2, label=label, alpha=0.9)

ax.axvline(n_dd, color='purple', linestyle='--', linewidth=1.5, alpha=0.5, label=f'$p = n = {n_dd}$')
ax.set_xlabel('Number of parameters $p$', fontsize=12)
ax.set_ylabel('Test MSE (log scale)', fontsize=12)
ax.set_title('Effect of Ridge Regularization on Double Descent', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
ax.set_xlim(1, max(p_values))

plt.tight_layout()
plt.show()

**Observations:**
- Without regularization ($\lambda = 0$), the test error spikes sharply at the interpolation threshold.
- Even a small amount of regularization ($\lambda = 0.01$) significantly dampens the peak.
- Stronger regularization ($\lambda = 1$) eliminates the peak entirely but may increase error in the overparametrized regime (too much bias).
- This illustrates the interplay: regularization trades off the variance peak for additional bias.

---
# 8. Mathematical Summary

## 8.1 The two U-curves

We have seen two related but distinct phenomena:

### Classical U-curve (Bias-Variance Tradeoff)

For a model class indexed by complexity parameter $d$ (e.g., polynomial degree):

$$\text{Test Error}(d) = \text{Bias}^2(d) + \text{Variance}(d) + \sigma^2$$

- $\text{Bias}^2(d)$ is monotonically decreasing in $d$ (richer models approximate $f$ better on average).
- $\text{Variance}(d)$ is monotonically increasing in $d$ (more parameters means more sensitivity to data).
- The sum produces a U-shape with a unique minimum.

### Modern Double Descent Curve

When we plot test error as a function of the number of parameters $p$ (crossing the interpolation threshold $p = n$):

$$\text{Test Error}(p) \sim \begin{cases}
\text{Classical U-shape} & p < n \text{ (underparametrized)} \\
\text{Peak} & p \approx n \text{ (interpolation threshold)} \\
\text{Decreasing} & p \gg n \text{ (overparametrized, benign overfitting)}
\end{cases}$$

## 8.2 Key mathematical insights

**Why does the peak occur at $p = n$?** The condition number of the Gram matrix $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ (or $\boldsymbol{\Phi}\boldsymbol{\Phi}^\top$) diverges as $p \to n$. This amplifies the noise component of $\mathbf{y}$, causing the solution $\hat{\boldsymbol{\beta}}$ to have large norm and wild oscillations.

**Why does overparametrization help?** In the overparametrized regime, the minimum-norm interpolator satisfies:

$$\|\hat{\boldsymbol{\beta}}\|^2 = \mathbf{y}^\top (\boldsymbol{\Phi}\boldsymbol{\Phi}^\top)^{-1} \mathbf{y}.$$

As $p$ grows (with suitable feature distributions), the eigenvalues of $\boldsymbol{\Phi}\boldsymbol{\Phi}^\top$ grow, making $(\boldsymbol{\Phi}\boldsymbol{\Phi}^\top)^{-1}$ smaller, hence $\|\hat{\boldsymbol{\beta}}\|$ smaller. The implicit $\ell^2$ regularization becomes stronger with more parameters.

## 8.3 Practical implications for deep learning

These observations have profound implications:

1. **"Bigger is better" (often):** Overparametrized networks are not just convenient — they can genuinely generalize better than smaller ones, provided they are trained properly.

2. **The interpolation threshold is dangerous:** If your network barely has enough capacity to fit the training data, it may perform *worse* than both a smaller and a larger network.

3. **Implicit regularization matters:** Gradient descent on overparametrized networks finds solutions with specific inductive biases (e.g., small norm, low rank) that promote generalization.

4. **Explicit regularization helps at the threshold:** Techniques like weight decay, dropout, and early stopping are most valuable near the interpolation threshold where the variance peak is sharpest.

---
# 9. Exercises

1. **(Bias-variance algebra)** Carry out the full proof of the bias-variance decomposition for an arbitrary loss function $\ell(y, \hat{y}) = (y - \hat{y})^2$, starting from $y = f(x) + \varepsilon$ with $\mathbb{E}[\varepsilon] = 0$ and $\text{Var}(\varepsilon) = \sigma^2$.

2. **(Code)** Modify the Monte Carlo simulation in Section 2.2 to use $f(x) = x^2 \sin(x)$ instead of $\sin(x)$. How does the optimal degree change? Does the U-curve shift?

3. **(Interpolation threshold)** For polynomial regression with $n$ data points and degree $d$, explain algebraically why the training error is exactly zero when $d \geq n - 1$ (assuming distinct $x_i$).

4. **(Condition number)** Generate the Vandermonde matrix for $n = 20$ equally spaced points and degrees $d = 5, 10, 15, 19, 25$. Compute the condition number $\kappa(\mathbf{V})$ in each case. What happens near $d = n - 1 = 19$?

5. **(Regularization)** In the Ridge regularization experiment (Section 7), explain mathematically why the regularized solution $\hat{\boldsymbol{\beta}}_{\text{ridge}} = (\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \lambda \mathbf{I})^{-1}\boldsymbol{\Phi}^\top \mathbf{y}$ remains bounded even when $\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$ is singular.

6. **(Graduate-level)** Prove that for the minimum-norm interpolator with i.i.d. Gaussian features $\boldsymbol{\phi}_j \sim \mathcal{N}(0, \mathbf{I}_n / p)$, the expected squared norm $\mathbb{E}[\|\hat{\boldsymbol{\beta}}\|^2]$ is finite for $p > n + 1$ and diverges as $p \to n^+$. (*Hint:* Use the Marchenko-Pastur distribution for the eigenvalues of $\boldsymbol{\Phi}\boldsymbol{\Phi}^\top$.)

---
# 10. References and Further Reading

- **Hastie, Tibshirani, Friedman.** *The Elements of Statistical Learning*, Ch. 7. (The classical bias-variance tradeoff.)
- **Belkin, Hsu, Ma, Mandal.** "Reconciling modern machine learning practice and the bias-variance trade-off." *PNAS* 116 (2019). (Double descent.)
- **Nakkiran, Kaplun, Bansal, Yang, Barak, Sutskever.** "Deep double descent: Where bigger models and more data can hurt." *ICLR* 2020.
- **Bartlett, Long, Lugosi, Tsigler.** "Benign overfitting in linear regression." *PNAS* 117 (2020).
- **Zhang, Bengio, Hardt, Recht, Vinyals.** "Understanding deep learning requires rethinking generalization." *ICLR* 2017. (Networks can memorize random labels.)
- **Hastie, Montanari, Rosset, Tibshirani.** "Surprises in high-dimensional ridgeless least squares interpolation." *Annals of Statistics* 50 (2022).

---
---